# ============================================
#  Notebook 04 — Source Document Text Extraction
#  Memorial Sloan Kettering | Goel Lab
# ============================================

# Notebook 4: Source Document Text Extraction & Processing

**Purpose:**
1. OCR-extract text from scanned source PDFs
2. Deidentify extracted text (using `deidentify_source_docs` patterns)
3. Maintain trackable `case_id <-> patient <-> document` mapping
4. Produce per-case text corpus for downstream feature extraction

**Outputs:**
- `extracted_text/` — one `.txt` per case (deidentified)
- `case_document_mapping.csv` — links case_id to original patient/doc
- `extraction_log.csv` — OCR quality stats per page

In [ ]:
import os
import re
import hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional

import fitz  # PyMuPDF
import pytesseract
import pandas as pd
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

PROJECT_ROOT = Path("/Users/robertjames/Documents/llm_summarization")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DEID_DIR = PROJECT_ROOT / "data" / "deidentified"
TEXT_DIR = PROJECT_ROOT / "data" / "extracted_text"
TEXT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 4.1 Case ID & Document Mapping

Each PDF gets a deterministic `case_id`. The mapping table records original filename, patient identifier (if parseable from filename), and document type (radiology/pathology).

In [ ]:
def generate_case_id(filename: str) -> str:
    h = hashlib.sha256(filename.encode()).hexdigest()[:12]
    return f"CASE_{h.upper()}"

def infer_doc_type(filename: str) -> str:
    fn = filename.lower()
    if any(kw in fn for kw in ["rad", "mammo", "mri", "ultrasound", "imaging"]):
        return "radiology"
    elif any(kw in fn for kw in ["path", "biopsy", "histol", "receptor"]):
        return "pathology"
    return "unknown"

def build_document_mapping(pdf_dir: Path) -> pd.DataFrame:
    rows = []
    for pdf_path in sorted(pdf_dir.rglob("*.pdf")):
        case_id = generate_case_id(pdf_path.stem)
        rows.append({
            "case_id": case_id,
            "original_filename": pdf_path.name,
            "original_stem": pdf_path.stem,
            "doc_type_inferred": infer_doc_type(pdf_path.name),
            "original_path": str(pdf_path),
            "n_pages": None,  # filled during extraction
        })
    return pd.DataFrame(rows)

mapping_df = build_document_mapping(RAW_DIR)
print(f"Documents found: {len(mapping_df)}")
mapping_df.head()

## 4.2 Redaction Patterns for Extracted Text

In [ ]:
@dataclass
class RedactionRule:
    name: str
    pattern: str
    flags: int = re.IGNORECASE

DEFAULT_TEXT_RULES = [
    RedactionRule("EMAIL", r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b"),
    RedactionRule("URL", r"\bhttps?://\S+\b|\bwww\.\S+\b"),
    RedactionRule("PHONE", r"\b(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"),
    RedactionRule("SSN", r"\b\d{3}-\d{2}-\d{4}\b"),
    RedactionRule("DATE_MDY", r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"),
    RedactionRule("DATE_TEXT", r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\s+\d{1,2},\s+\d{4}\b"),
    RedactionRule("ZIP", r"\b\d{5}(?:-\d{4})?\b"),
    RedactionRule("MRN_CONTEXT", r"(?:MRN|Medical Record|Patient ID)[:\s]*\S+"),
    RedactionRule("NAME_CONTEXT", r"(?:Patient|Name|DOB)[:\s]+[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3}"),
]

def compile_text_rules(rules):
    return [(r.name, re.compile(r.pattern, r.flags)) for r in rules]

def redact_text(text: str, compiled_rules) -> str:
    out = text
    for name, pat in compiled_rules:
        out = pat.sub(f"[{name}]", out)
    return out

compiled_rules = compile_text_rules(DEFAULT_TEXT_RULES)
print(f"Loaded {len(compiled_rules)} redaction rules")

## 4.3 OCR Extraction + Deidentification Pipeline

In [ ]:
def render_page_to_pil(doc, page_index, dpi=300):
    page = doc.load_page(page_index)
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

def extract_and_deidentify_pdf(pdf_path: Path, compiled_rules, dpi=300, lang="eng"):
    """OCR a PDF, redact PHI from text, return per-page results."""
    doc = fitz.open(pdf_path)
    pages = []
    for p in range(doc.page_count):
        pil_img = render_page_to_pil(doc, p, dpi)
        # OCR full text
        raw_text = pytesseract.image_to_string(pil_img, lang=lang)
        # OCR confidence
        ocr_data = pytesseract.image_to_data(pil_img, lang=lang, output_type=pytesseract.Output.DATAFRAME)
        ocr_data["conf"] = pd.to_numeric(ocr_data["conf"], errors="coerce")
        avg_conf = ocr_data["conf"].dropna().mean()
        n_words = int(ocr_data["text"].dropna().apply(lambda x: len(str(x).strip()) > 0).sum())
        # Deidentify
        clean_text = redact_text(raw_text, compiled_rules)
        pages.append({
            "page": p + 1,
            "raw_char_count": len(raw_text),
            "clean_char_count": len(clean_text),
            "n_words": n_words,
            "avg_ocr_confidence": avg_conf,
            "clean_text": clean_text,
        })
    doc.close()
    return pages

In [ ]:
# Run extraction on all PDFs
log_rows = []
for i, row in tqdm(mapping_df.iterrows(), total=len(mapping_df), desc="Extracting text"):
    pdf_path = Path(row["original_path"])
    case_id = row["case_id"]
    if not pdf_path.exists():
        log_rows.append({"case_id": case_id, "file": pdf_path.name, "status": "FILE_NOT_FOUND"})
        continue
    try:
        pages = extract_and_deidentify_pdf(pdf_path, compiled_rules)
    except Exception as e:
        log_rows.append({"case_id": case_id, "file": pdf_path.name, "status": "ERROR", "error": str(e)})
        continue

    # Save combined text
    full_text = "\n\n--- PAGE BREAK ---\n\n".join(p["clean_text"] for p in pages)
    txt_path = TEXT_DIR / f"{case_id}.txt"
    txt_path.write_text(full_text, encoding="utf-8")

    # Update mapping with page count
    mapping_df.loc[i, "n_pages"] = len(pages)

    # Log per-page stats
    for p in pages:
        log_rows.append({
            "case_id": case_id,
            "file": pdf_path.name,
            "page": p["page"],
            "status": "OK",
            "raw_chars": p["raw_char_count"],
            "clean_chars": p["clean_char_count"],
            "n_words": p["n_words"],
            "avg_ocr_confidence": p["avg_ocr_confidence"],
        })

df_log = pd.DataFrame(log_rows)
df_log.to_csv(OUTPUT_DIR / "text_extraction_log.csv", index=False)
mapping_df.to_csv(DEID_DIR / "case_document_mapping.csv", index=False)
print(f"Extracted {len(mapping_df)} documents -> {TEXT_DIR}")
print(f"Extraction log: {len(df_log)} entries")

## 4.4 Extraction Quality Summary

In [ ]:
ok_pages = df_log[df_log["status"] == "OK"]
print(f"Total pages extracted: {len(ok_pages)}")
print(f"Average OCR confidence: {ok_pages['avg_ocr_confidence'].mean():.1f}%")
print(f"Average words per page: {ok_pages['n_words'].mean():.0f}")
print(f"Average chars per page (after redaction): {ok_pages['clean_chars'].mean():.0f}")

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(ok_pages["avg_ocr_confidence"].dropna(), bins=30, color="#3498db", edgecolor="black")
axes[0].set_xlabel("Avg OCR Confidence (%)")
axes[0].set_title("OCR Confidence Distribution")

axes[1].hist(ok_pages["n_words"], bins=30, color="#2ecc71", edgecolor="black")
axes[1].set_xlabel("Words per Page")
axes[1].set_title("Words per Page Distribution")

axes[2].hist(ok_pages["clean_chars"], bins=30, color="#e74c3c", edgecolor="black")
axes[2].set_xlabel("Characters (after redaction)")
axes[2].set_title("Characters per Page Distribution")

plt.suptitle("Text Extraction Quality Summary", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "text_extraction_quality.png", dpi=300)
plt.show()

## 4.5 Per-Case Text Summary

In [ ]:
# Aggregate per-case stats
case_stats = (
    ok_pages.groupby("case_id")
    .agg(
        n_pages=("page", "count"),
        total_words=("n_words", "sum"),
        avg_confidence=("avg_ocr_confidence", "mean"),
        total_clean_chars=("clean_chars", "sum"),
    )
    .reset_index()
)
case_stats = case_stats.merge(mapping_df[["case_id", "doc_type_inferred"]], on="case_id", how="left")

print(f"Per-case summary: {len(case_stats)} cases")
case_stats.describe()

In [ ]:
# Save
case_stats.to_csv(OUTPUT_DIR / "per_case_text_stats.csv", index=False)
print("\nSample extracted text (first 500 chars of first case):")
sample_file = sorted(TEXT_DIR.glob("*.txt"))
if sample_file:
    print(sample_file[0].read_text(encoding="utf-8")[:500])

---
## 4.6 Alternative Extraction Methods

We compare three text extraction pipelines on the same scanned PDFs to evaluate which method produces higher-quality text for downstream clinical feature extraction:

| Method | Engine | Approach |
|--------|--------|----------|
| **Method 1: pytesseract OCR** | Tesseract 5 | Traditional OCR — rasterize page → run Tesseract → raw text |
| **Method 2: Claude Vision API** | Anthropic Claude 3 | Send page image to Claude vision endpoint with a description prompt |
| **Method 3: Claude Transcription API** | Anthropic Claude 3 | Send page image with an explicit "transcribe exactly" instruction |

All methods produce per-page text that is then deidentified using the same redaction rules (Section 4.2).

**Requires:** `pip install anthropic` and a valid `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import base64
import os
import time

from anthropic import Anthropic

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    print("WARNING: ANTHROPIC_API_KEY not set. Claude-based extraction methods will be skipped.")
    claude_available = False
else:
    client = Anthropic(api_key=ANTHROPIC_API_KEY)
    CLAUDE_MODEL = "claude-sonnet-4-20250514"
    claude_available = True
    print(f"Anthropic client initialized (model: {CLAUDE_MODEL})")

### Method 2: Claude Vision API — Descriptive Extraction

Sends each page image to the Claude Vision endpoint with a prompt asking it to describe and extract all text content from the clinical document image. This leverages Claude's multimodal understanding to interpret document layout, tables, and handwriting.

In [ ]:
def pil_image_to_base64(pil_img: Image.Image, fmt="PNG") -> str:
    """Convert a PIL image to a base64-encoded string."""
    import io
    buf = io.BytesIO()
    pil_img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def extract_with_claude_vision(pdf_path: Path, compiled_rules, dpi=300) -> list:
    """Extract text from a PDF using Claude Vision API (descriptive extraction).
    
    Each page is rendered to an image, sent to Claude with a prompt asking it
    to extract all visible text content from the clinical document.
    """
    doc = fitz.open(pdf_path)
    pages = []
    for p in range(doc.page_count):
        pil_img = render_page_to_pil(doc, p, dpi)
        b64_str = pil_image_to_base64(pil_img, fmt="PNG")
        
        message_list = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {"type": "base64", "media_type": "image/png", "data": b64_str},
                    },
                    {
                        "type": "text",
                        "text": (
                            "This is a scanned clinical document (radiology or pathology report). "
                            "Extract ALL visible text from this image. Preserve the original layout, "
                            "headings, labels, and values as faithfully as possible. "
                            "Output only the extracted text, nothing else."
                        ),
                    },
                ],
            }
        ]
        
        try:
            response = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=4096, messages=message_list
            )
            raw_text = response.content[0].text
        except Exception as e:
            raw_text = f"[CLAUDE_VISION_ERROR: {e}]"
        
        clean_text = redact_text(raw_text, compiled_rules)
        pages.append({
            "page": p + 1,
            "raw_char_count": len(raw_text),
            "clean_char_count": len(clean_text),
            "n_words": len(raw_text.split()),
            "method": "claude_vision",
            "clean_text": clean_text,
        })
        time.sleep(0.5)  # rate limiting
    doc.close()
    return pages

print("Claude Vision extraction function defined.")

### Method 3: Claude Transcription API — Exact Transcription

Sends each page image to Claude with an explicit "transcribe exactly" instruction, optimized for faithful reproduction of typed and handwritten text, forms, tables, and structured clinical data. This mirrors the approach in `how_to_transcribe_text.ipynb`.

In [ ]:
def extract_with_claude_transcription(pdf_path: Path, compiled_rules, dpi=300) -> list:
    """Extract text from a PDF using Claude Transcription API (exact transcription).
    
    Each page is rendered to an image and sent to Claude with an instruction
    to transcribe exactly all text visible in the image — typed, handwritten,
    tabular, and structured content.
    """
    doc = fitz.open(pdf_path)
    pages = []
    for p in range(doc.page_count):
        pil_img = render_page_to_pil(doc, p, dpi)
        b64_str = pil_image_to_base64(pil_img, fmt="PNG")
        
        message_list = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {"type": "base64", "media_type": "image/png", "data": b64_str},
                    },
                    {
                        "type": "text",
                        "text": (
                            "Transcribe all text in this image exactly as it appears. "
                            "Include all headings, labels, values, table contents, "
                            "handwritten notes, stamps, and any other visible text. "
                            "Preserve the structure and formatting. "
                            "Output only the transcribed text and nothing else."
                        ),
                    },
                ],
            }
        ]
        
        try:
            response = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=4096, messages=message_list
            )
            raw_text = response.content[0].text
        except Exception as e:
            raw_text = f"[CLAUDE_TRANSCRIPTION_ERROR: {e}]"
        
        clean_text = redact_text(raw_text, compiled_rules)
        pages.append({
            "page": p + 1,
            "raw_char_count": len(raw_text),
            "clean_char_count": len(clean_text),
            "n_words": len(raw_text.split()),
            "method": "claude_transcription",
            "clean_text": clean_text,
        })
        time.sleep(0.5)  # rate limiting
    doc.close()
    return pages

print("Claude Transcription extraction function defined.")

---
## 4.7 Run All Three Extraction Methods & Compare

Run each method on the same set of PDFs. Store results in a unified DataFrame for head-to-head comparison on word count, character count, and downstream extraction quality.

In [ ]:
METHODS = {
    "pytesseract": extract_and_deidentify_pdf,
}

if claude_available:
    METHODS["claude_vision"] = extract_with_claude_vision
    METHODS["claude_transcription"] = extract_with_claude_transcription

COMPARISON_DIR = PROJECT_ROOT / "data" / "extracted_text_comparison"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

all_method_rows = []

for method_name, extract_fn in METHODS.items():
    print(f"\n{'='*60}")
    print(f"Running extraction method: {method_name}")
    print(f"{'='*60}")
    
    method_dir = COMPARISON_DIR / method_name
    method_dir.mkdir(parents=True, exist_ok=True)
    
    for i, row in tqdm(mapping_df.iterrows(), total=len(mapping_df), desc=method_name):
        pdf_path = Path(row["original_path"])
        case_id = row["case_id"]
        if not pdf_path.exists():
            all_method_rows.append({
                "case_id": case_id, "method": method_name,
                "file": pdf_path.name, "status": "FILE_NOT_FOUND",
            })
            continue
        try:
            pages = extract_fn(pdf_path, compiled_rules)
        except Exception as e:
            all_method_rows.append({
                "case_id": case_id, "method": method_name,
                "file": pdf_path.name, "status": "ERROR", "error": str(e),
            })
            continue

        # Save combined text per method
        full_text = "\n\n--- PAGE BREAK ---\n\n".join(p["clean_text"] for p in pages)
        (method_dir / f"{case_id}.txt").write_text(full_text, encoding="utf-8")

        for p in pages:
            all_method_rows.append({
                "case_id": case_id,
                "method": method_name,
                "file": pdf_path.name,
                "page": p["page"],
                "status": "OK",
                "raw_chars": p["raw_char_count"],
                "clean_chars": p["clean_char_count"],
                "n_words": p.get("n_words", 0),
                "avg_ocr_confidence": p.get("avg_ocr_confidence", np.nan),
            })

df_comparison = pd.DataFrame(all_method_rows)
df_comparison.to_csv(OUTPUT_DIR / "extraction_method_comparison.csv", index=False)
print(f"\nComparison results: {len(df_comparison)} rows across {len(METHODS)} methods")

---
## 4.8 Extraction Method Comparison — Summary Statistics & Visualizations

Compare the three methods across key quality metrics: word count, character count, and OCR confidence (where available). Inspired by the comparative trend visualization approach in *Additive Models for Prediction*.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
matplotlib.rcParams["axes.labelsize"] = 13
matplotlib.rcParams["xtick.labelsize"] = 11
matplotlib.rcParams["ytick.labelsize"] = 11
matplotlib.rcParams["text.color"] = "k"

ok_comp = df_comparison[df_comparison["status"] == "OK"].copy()

# --- Per-method summary table ---
method_summary = (
    ok_comp.groupby("method")
    .agg(
        n_pages=("page", "count"),
        mean_words=("n_words", "mean"),
        median_words=("n_words", "median"),
        mean_chars=("clean_chars", "mean"),
        median_chars=("clean_chars", "median"),
        mean_ocr_conf=("avg_ocr_confidence", "mean"),
    )
    .round(1)
)
print("Extraction Method Summary:")
display(method_summary)

In [ ]:
# --- Figure 1: Side-by-side boxplots of word count by method ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=ok_comp, x="method", y="n_words", palette="Set2", ax=axes[0], showmeans=True)
axes[0].set_title("Words per Page by Extraction Method")
axes[0].set_xlabel("Method")
axes[0].set_ylabel("Words per Page")

sns.boxplot(data=ok_comp, x="method", y="clean_chars", palette="Set2", ax=axes[1], showmeans=True)
axes[1].set_title("Characters per Page (after redaction)")
axes[1].set_xlabel("Method")
axes[1].set_ylabel("Characters per Page")

# Per-case total words
case_totals = ok_comp.groupby(["method", "case_id"]).agg(total_words=("n_words", "sum")).reset_index()
sns.boxplot(data=case_totals, x="method", y="total_words", palette="Set2", ax=axes[2], showmeans=True)
axes[2].set_title("Total Words per Case by Method")
axes[2].set_xlabel("Method")
axes[2].set_ylabel("Total Words per Case")

plt.suptitle("Extraction Method Comparison — Text Volume", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "extraction_method_boxplots.png", dpi=300, bbox_inches="tight")
plt.show()

---
## 4.9 Prompt History & Metric Trends per Extraction Method

Track how diagnostic metrics (accuracy, fabrication rate, omission rate) change across prompt versions and extraction methods. Inspired by the time-series trend + changepoint visualization approach in *Additive Models for Prediction*.

This section loads the prompt library to build a version timeline, then overlays per-method metric trajectories.

In [ ]:
# --- Load prompt library for version timeline ---
PROMPT_LIB_PATH = PROJECT_ROOT / "data" / "processed" / "prompt_library_updated_v5.xlsx"
if PROMPT_LIB_PATH.exists():
    prompt_lib = pd.read_excel(PROMPT_LIB_PATH)
    print(f"Prompt library loaded: {prompt_lib.shape}")
    display(prompt_lib.head())
else:
    # Create a synthetic prompt version timeline for illustration
    prompt_lib = pd.DataFrame({
        "prompt_version": [
            "zero_shot_structured_extraction_prod",
            "chain_of_thought_3",
            "rag_v1",
            "rag_v2",
            "few_shot_v1",
            "program_aided_v1",
            "react_v1",
            "2pop_mcode_gpt",
            "bfop_v1",
        ],
        "version_order": range(1, 10),
    })
    print("Using synthetic prompt version timeline (prompt library not found)")

# --- Simulated metric trajectories per prompt version per method ---
# In production, these would come from re-running extraction + scoring per prompt version.
# Here we build the scaffold so the plots are ready once real data is available.
np.random.seed(42)
n_versions = len(prompt_lib)
methods_list = list(METHODS.keys())

metric_trajectory_rows = []
for method in methods_list:
    base_acc = 0.78 if method == "pytesseract" else (0.85 if method == "claude_vision" else 0.87)
    base_fab = 0.12 if method == "pytesseract" else (0.07 if method == "claude_vision" else 0.05)
    base_omit = 0.10 if method == "pytesseract" else (0.08 if method == "claude_vision" else 0.08)
    for i in range(n_versions):
        noise = np.random.normal(0, 0.02)
        trend = i * 0.005  # slight improvement over prompt versions
        metric_trajectory_rows.append({
            "prompt_version": prompt_lib.iloc[i]["prompt_version"] if "prompt_version" in prompt_lib.columns else f"v{i+1}",
            "version_order": i + 1,
            "method": method,
            "accuracy": np.clip(base_acc + trend + noise, 0, 1),
            "fabrication_rate": np.clip(base_fab - trend * 0.5 + noise * 0.3, 0, 1),
            "omission_rate": np.clip(base_omit - trend * 0.3 + noise * 0.2, 0, 1),
        })

df_trajectory = pd.DataFrame(metric_trajectory_rows)
print(f"\nMetric trajectory table: {df_trajectory.shape}")
df_trajectory.head()

In [ ]:
# --- Figure 2: Metric Trends across Prompt Versions (Additive Models style) ---
colors = {"pytesseract": "b", "claude_vision": "r", "claude_transcription": "gold"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for metric, ax, title in zip(
    ["accuracy", "fabrication_rate", "omission_rate"],
    axes,
    ["Accuracy", "Fabrication Rate", "Omission Rate"],
):
    for method in methods_list:
        subset = df_trajectory[df_trajectory["method"] == method].sort_values("version_order")
        ax.plot(
            subset["version_order"], subset[metric],
            color=colors.get(method, "grey"), marker="o", linewidth=2, label=method
        )
    ax.set_xlabel("Prompt Version (chronological)")
    ax.set_ylabel(title)
    ax.set_title(f"{title} across Prompt Versions")
    ax.legend(loc="best", fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Prompt History: Change in Metrics per Extraction Method",
    fontsize=15, fontweight="bold"
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "prompt_history_metric_trends.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure 3: Forecast-style comparison with uncertainty bounds (Additive Models style) ---
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for method in methods_list:
    subset = df_trajectory[df_trajectory["method"] == method].sort_values("version_order")
    y = subset["accuracy"].values
    x = subset["version_order"].values
    # Simulate uncertainty bounds (in production, use bootstrap CIs)
    upper = np.clip(y + 0.04, 0, 1)
    lower = np.clip(y - 0.04, 0, 1)
    
    color = colors.get(method, "grey")
    ax.plot(x, y, color=color, linewidth=2, label=f"{method} accuracy")
    ax.fill_between(x, upper, lower, alpha=0.2, color=color, edgecolor="k", linewidth=0.5)

ax.set_xlabel("Prompt Version (chronological)")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy Forecast with Uncertainty Bounds per Extraction Method", fontsize=14)
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "accuracy_forecast_uncertainty.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# --- Save all comparison outputs ---
df_trajectory.to_csv(OUTPUT_DIR / "prompt_metric_trajectories.csv", index=False)

print("=" * 60)
print("OUTPUTS SAVED")
print("=" * 60)
for f in [
    "extraction_method_comparison.csv",
    "extraction_method_boxplots.png",
    "prompt_history_metric_trends.png",
    "accuracy_forecast_uncertainty.png",
    "prompt_metric_trajectories.csv",
]:
    fpath = OUTPUT_DIR / f
    status = f"{fpath.stat().st_size / 1024:.1f} KB" if fpath.exists() else "PENDING (run cells above)"
    print(f"  {f:50s} {status}")